# Sub-topic modeling: LDA (paragraph level)

## Setup

In [ ]:
import pandas as pd
import numpy as np
import csv
import os
import random
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import warnings
import spacy_stanza
from sklearn.feature_extraction.text import TfidfVectorizer
# from gensim import matutils
# from gensim.models import CoherenceModel, LdaModel

import nltk
nltk.download('stopwords')
stopwords = nltk.corpus.stopwords.words('dutch')

# Load Datasets

In [ ]:
# articles df
articles_df = pd.read_csv('data/NOS_final_topics.csv',
                          sep = ';', encoding = 'utf-8', quoting=csv.QUOTE_NONNUMERIC)
print(articles_df.shape)

articles_df['article_id'] = articles_df['article_id'].astype(int)

articles_df['Text'] = articles_df['Text'].str.replace('[LINE_BREAK]', '\n')

In [ ]:
# get the minimum and maximum date
print(articles_df['Date'].min())
print(articles_df['Date'].max())

In [ ]:
df_covid = articles_df[articles_df['about_covid'] == 1]
print(df_covid.shape)

In [ ]:
df_covid['paragraphs'] = df_covid['Text'].str.split('\n')
print(df_covid.shape)

In [ ]:
# make the df paragraph level, so each instance of the paragraphs list gets a new row
df_covid_par = df_covid.explode('paragraphs')
print(df_covid_par.shape)

In [ ]:
tokenizer = spacy_stanza.load_pipeline("nl", processors="tokenize")
lemmatizer = spacy_stanza.load_pipeline("nl", processors="tokenize,lemma")

In [ ]:
import re

def clean_paragraphs(par,
                     tokenizer,
                     lemmatizer,
                     stopwords=None,
                     keep_hyphen=False):
    """
    Tokenize -> lowercase -> remove special chars -> remove stopwords -> lemmatize.
    Returns cleaned string (lemmas) or "" for non-string inputs.
    """
    if not isinstance(par, str):
        return ""

    stop_set = set(w.lower() for w in (stopwords or []))

    # allowed char class: letters + accented; optionally keep hyphen
    char_class = "a-zA-ZÀ-ÿ" + (r"\-" if keep_hyphen else "")
    non_allowed_re = re.compile(fr"[^{char_class}]", flags=re.UNICODE)

    # 1) tokenize and lowercase token texts
    raw_tokens = [t.text for t in tokenizer(par)]

    # 2-4) clean tokens and filter stopwords/short tokens
    cleaned_tokens = []
    for raw in raw_tokens:
        t = raw.lower().strip()
        t = non_allowed_re.sub("", t)          # remove digits/punct/special
        if t in stop_set:
        cleaned_tokens.append(t)

    if not cleaned_tokens:
        return ""

    # 5) lemmatize: pass cleaned text to the lemmatizer and keep cleaned lemmas
    cleaned_text = " ".join(cleaned_tokens)
    lemmas = []
    for tok in lemmatizer(cleaned_text):
        lemma = getattr(tok, "lemma_", "") or ""
        lemma = lemma.lower().strip()
        lemma = non_allowed_re.sub("", lemma)

        if lemma in stop_set:
        lemmas.append(lemma)

    return " ".join(lemmas)

In [ ]:
# read df_covid_par
df_covid_par = pd.read_csv('data/df_covid_paragraphs_cleaned.csv', sep = ';', encoding = 'utf-8', quoting=csv.QUOTE_NONNUMERIC)
print(df_covid_par.shape)   

In [ ]:
# calculate length of cleaned paragraphs
df_covid_par['par_len'] = df_covid_par['paragraphs'].str.split().apply(len)

In [ ]:
df_covid_par[df_covid_par['par_len'] < 15]['paragraphs'].values

In [ ]:
# drop if par_len is less than 15
df_covid_par = df_covid_par[df_covid_par['par_len'] >= 15]
print(df_covid_par.shape)

In [ ]:
# see a random selection of 5 paragraphs and cleaned versions
for i in random.sample(range(df_covid_par.shape[0]), 5):
    print("Original paragraph:")
    print("Cleaned paragraph:")
    print()

In [ ]:
# see paragraph values where length is less than 16
df_covid_par[df_covid_par['par_len'] < 20]['cleaned_par'].values

In [ ]:
# relative pruning: removing all terms that occurred in more than 99% or less than .5% of all documents
from sklearn.feature_extraction.text import CountVectorizer

# docs: an iterable of preprocessed strings (e.g. df_covid_par['cleaned_par'])
docs = df_covid_par['cleaned_par'].fillna("").astype(str).tolist()
n_docs = len(docs)

# unique tokens before pruning
cv_all = CountVectorizer(token_pattern=r"(?u)\b\w+\b", lowercase=False, stop_words=stopwords)
dtm_all = cv_all.fit_transform(docs)   # sparse (n_docs, n_terms)
all_features = cv_all.get_feature_names_out()
print(f"Documents: {n_docs}, unique tokens before pruning: {len(all_features)}")

In [ ]:
# Keep tokens that appear in at least 0.1% of docs and drop tokens appearing in >70% of docs
cv = CountVectorizer(min_df=0.001,   
                     max_df=0.70,    
                     token_pattern=r"(?u)\b\w+\b",  # adjust token pattern if needed
                     lowercase=False,
                     stop_words=stopwords)               # False if your cleaned text is already lowercased

dtm = cv.fit_transform(docs)   # sparse (n_docs, n_terms_kept)
features = cv.get_feature_names_out()

print(f"Documents: {n_docs}, unique tokens after pruning: {len(features)}")
print("Sample features:", features[:30])

In [ ]:
corpus = matutils.Sparse2Corpus(dtm, documents_columns=False)
vocab = dict(enumerate(cv.get_feature_names_out()))

In [ ]:
# For coherence: create separate gensim Dictionary
import gensim.corpora as corpora
analyzer = cv.build_analyzer()
tokens = [analyzer(doc) for doc in docs]
dictionary = corpora.Dictionary(tokens)

In [ ]:
# pruning 0.001
result = []
for k in [5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 75, 80]:
    m = LdaModel(
        corpus,
        num_topics=k,
        id2word=vocab,
        random_state=42,
        alpha="asymmetric",
    )
    perplexity = m.log_perplexity(corpus)
    coherence = CoherenceModel(
        model=m, corpus=corpus, coherence="u_mass"
    ).get_coherence()
    result.append(dict(k=k, perplexity=perplexity, coherence=coherence))

result = pd.DataFrame(result)
result.plot(x="k", y=["perplexity", "coherence"], xticks=result['k'])
plt.show()

In [ ]:
lda_25 = LdaModel(
    corpus, id2word=vocab, num_topics=25, random_state=42, alpha="asymmetric"
)

In [ ]:
# save the LDA topic model
lda_25.save("covidlda_model/covidlda_25.model")

In [ ]:
coherence_umass = CoherenceModel(
    model=lda_25, 
    corpus=corpus, 
    coherence="u_mass"
).get_coherence()
print(f"LDA 25 u_mass Coherence score: {coherence_umass}")

In [ ]:
coherence_cnpmi = CoherenceModel(
    model=lda_25, 
    texts=tokens,
    dictionary=dictionary,  # Gensim Dictionary object
    coherence="c_npmi"
).get_coherence()
print(f"LDA 25 c_npmi Coherence score: {coherence_cnpmi}")

In [ ]:
coherence_cv = CoherenceModel(
    model=lda_25, 
    texts=tokens,
    dictionary=dictionary,  # Gensim Dictionary object
    coherence="c_v"
).get_coherence()
print(f"LDA 25 c_v Coherence score: {coherence_cv}")

In [ ]:
pd.DataFrame(
    {
        f"Topic {n}": [w for (w, tw) in words]
        for (n, words) in lda_25.show_topics(formatted=False, num_words=10, num_topics=25)
    }
)

In [ ]:
# make this a df
topic_words = pd.DataFrame(
    {
        f"Topic {n}": [w for (w, tw) in words]
        for (n, words) in lda_25.show_topics(formatted=False, num_words=10, num_topics=25)
    }
)

topic_words=topic_words.transpose().reset_index().rename(columns={'index': 'topic'})

# make the words one list per topic
topic_words['words'] = topic_words.apply(lambda row: ', '.join(row[1:].values), axis=1)
topic_words = topic_words[['topic', 'words']]

In [ ]:
# add topics to df
topics = pd.DataFrame(
    [
        dict(lda_25.get_document_topics(doc, minimum_probability=0.0))
        for doc in corpus
    ]
)
print(topics.shape)
topics.columns = [f'Top_{i+1}' for i in range(topics.shape[1])]

In [ ]:
# combine topics with df_covid_par
df_covid_par_topics25 = pd.concat([df_covid_par.reset_index(drop=True), topics.reset_index(drop=True)], axis=1)

# get the max topic per paragraph
df_covid_par_topics25['max_topic'] = df_covid_par_topics25[[f'Top_{i+1}' for i in range(lda_25.num_topics)]].idxmax(axis=1)

In [ ]:
for i in df_covid_par_topics25.columns:
    print(i)

# drop all columns that start with subtopic_ and Top_ keep only max_topic
df_covid_par_topics25 = df_covid_par_topics25.loc[:, ~df_covid_par_topics25.columns.str.startswith('subtopic_')]
df_covid_par_topics25 = df_covid_par_topics25.loc[:, ~df_covid_par_topics25.columns.str.startswith('Top_')]

# rename max_topic to LDA25_Topic
df_covid_par_topics25 = df_covid_par_topics25.rename(columns={'max_topic': 'LDA25_Topic'})

In [ ]:
# save this
df_covid_par_topics25.to_csv('data/df_covid_paragraphs_with_topics_25.csv', sep = ';', encoding = 'utf-8', quoting=csv.QUOTE_NONNUMERIC, index=False)

In [ ]:
# add paragraphs to topics df
topics['paragraph'] = df_covid_par['paragraphs'].values

In [ ]:
top_paragraphs = {}
for topic in range(lda_25.num_topics):
    col = f'Top_{topic+1}'
    top_paragraphs[topic] = topics.nlargest(5, col)[['paragraph', col]].values.tolist()

In [ ]:
topic_metadata = pd.DataFrame.from_dict(top_paragraphs, orient='index').transpose()

topic_metadata.columns = [f'Top_{i+1}' for i in range(topic_metadata.shape[1])]

In [ ]:
# transpose again to make topics the rows and columns top paragraphs and their probabilities for each paragraph
topic_metadata = topic_metadata.transpose()

topic_metadata['paragraphs'] = topic_metadata.apply(lambda row: [item[0] for item in row], axis=1)
topic_metadata['probabilities'] = topic_metadata.apply(lambda row: [item[1] for item in row], axis=1)
topic_metadata = topic_metadata[['paragraphs', 'probabilities']]
topic_metadata = topic_metadata.reset_index().rename(columns={'index': 'topic'})

# Flatten the dictionary into a list of rows
rows = []
for topic, para_probs in top_paragraphs.items():
    for para, prob in para_probs:
        rows.append({'topic': topic, 'paragraph': para, 'probability': prob})

# Create DataFrame
topic_metadata = pd.DataFrame(rows)

In [ ]:
# add the topic values Topic to be able to match with topic words
topic_metadata['topic'] = topic_metadata['topic'].apply(lambda x: f'Topic {x}')

In [ ]:
# match with topic words
topic_metadata_merged = topic_metadata.merge(topic_words, on='topic', how='left')

In [ ]:
topic_metadata_merged.to_csv('data/covidlda_25_metadata.csv', sep = ';', encoding = 'utf-8', quoting=csv.QUOTE_NONNUMERIC, index=False)

# Run LDA with 50 Topics 0.1% min_df

In [ ]:
lda_50 = LdaModel(
    corpus, id2word=vocab, num_topics=50, random_state=42, alpha="asymmetric"
)

In [ ]:
# save the LDA topic model
lda_50.save("covidlda_model/covidlda_50.model")

In [ ]:
coherence_umass = CoherenceModel(
    model=lda_50, 
    corpus=corpus, 
    coherence="u_mass"
).get_coherence()
print(f"LDA 50 u_mass Coherence score: {coherence_umass}")

In [ ]:
coherence_cnpmi = CoherenceModel(
    model=lda_50, 
    texts=tokens,
    dictionary=dictionary,  # Gensim Dictionary object
    coherence="c_npmi"
).get_coherence()
print(f"LDA 50 c_npmi Coherence score: {coherence_cnpmi}")

In [ ]:
coherence_cv = CoherenceModel(
    model=lda_50, 
    texts=tokens,
    dictionary=dictionary,  # Gensim Dictionary object
    coherence="c_v"
).get_coherence()
print(f"LDA 50 c_v Coherence score: {coherence_cv}")

In [ ]:
pd.DataFrame(
    {
        f"Topic {n}": [w for (w, tw) in words]
        for (n, words) in lda_50.show_topics(formatted=False, num_words=10, num_topics=50)
    }
)

In [ ]:
# make this a df
topic_words = pd.DataFrame(
    {
        f"Topic {n}": [w for (w, tw) in words]
        for (n, words) in lda_50.show_topics(formatted=False, num_words=10, num_topics=50)
    }
)

topic_words=topic_words.transpose().reset_index().rename(columns={'index': 'topic'})

# make the words one list per topic
topic_words['words'] = topic_words.apply(lambda row: ', '.join(row[1:].values), axis=1)
topic_words = topic_words[['topic', 'words']]

In [ ]:
# add topics to df
topics = pd.DataFrame(
    [
        dict(lda_50.get_document_topics(doc, minimum_probability=0.0))
        for doc in corpus
    ]
)
print(topics.shape)
topics.columns = [f'Top_{i+1}' for i in range(topics.shape[1])]

In [ ]:
# combine topics with df_covid_par
df_covid_par_topics50 = pd.concat([df_covid_par.reset_index(drop=True), topics.reset_index(drop=True)], axis=1)

# get the max topic per paragraph
df_covid_par_topics50['max_topic'] = df_covid_par_topics50[[f'Top_{i+1}' for i in range(lda_50.num_topics)]].idxmax(axis=1)

In [ ]:
for i in df_covid_par_topics50.columns:
    print(i)

# drop all columns that start with subtopic_ and Top_ keep only max_topic
df_covid_par_topics50 = df_covid_par_topics50.loc[:, ~df_covid_par_topics50.columns.str.startswith('subtopic_')]
df_covid_par_topics50 = df_covid_par_topics50.loc[:, ~df_covid_par_topics50.columns.str.startswith('Top_')]

# rename max_topic to LDA50_Topic
df_covid_par_topics50 = df_covid_par_topics50.rename(columns={'max_topic': 'LDA50_Topic'})

In [ ]:
# save this
df_covid_par_topics50.to_csv('data/df_covid_paragraphs_with_topics_50.csv', sep = ';', encoding = 'utf-8', quoting=csv.QUOTE_NONNUMERIC, index=False)

In [ ]:
# add paragraphs to topics df
topics['paragraph'] = df_covid_par['paragraphs'].values

In [ ]:
top_paragraphs = {}
for topic in range(lda_50.num_topics):
    col = f'Top_{topic+1}'
    top_paragraphs[topic] = topics.nlargest(5, col)[['paragraph', col]].values.tolist()

In [ ]:
topic_metadata = pd.DataFrame.from_dict(top_paragraphs, orient='index').transpose()

topic_metadata.columns = [f'Top_{i+1}' for i in range(topic_metadata.shape[1])]

In [ ]:
# transpose again to make topics the rows and columns top paragraphs and their probabilities for each paragraph
topic_metadata = topic_metadata.transpose()

topic_metadata['paragraphs'] = topic_metadata.apply(lambda row: [item[0] for item in row], axis=1)
topic_metadata['probabilities'] = topic_metadata.apply(lambda row: [item[1] for item in row], axis=1)
topic_metadata = topic_metadata[['paragraphs', 'probabilities']]
topic_metadata = topic_metadata.reset_index().rename(columns={'index': 'topic'})

# Flatten the dictionary into a list of rows
rows = []
for topic, para_probs in top_paragraphs.items():
    for para, prob in para_probs:
        rows.append({'topic': topic, 'paragraph': para, 'probability': prob})

# Create DataFrame
topic_metadata = pd.DataFrame(rows)

In [ ]:
# add the topic values Topic to be able to match with topic words
topic_metadata['topic'] = topic_metadata['topic'].apply(lambda x: f'Topic {x}')

In [ ]:
# match with topic words
topic_metadata_merged = topic_metadata.merge(topic_words, on='topic', how='left')

In [ ]:
topic_metadata_merged.to_csv('data/covidlda_50_metadata.csv', sep = ';', encoding = 'utf-8', quoting=csv.QUOTE_NONNUMERIC, index=False)

# Read the metadata manually annotated and match with articles

In [ ]:
df_covid = pd.read_csv('data/df_covid_paragraphs_with_topics_50.csv', sep = ';', encoding = 'utf-8', quoting=csv.QUOTE_NONNUMERIC)
print(df_covid.shape)
df_covid['article_id'] = df_covid['article_id'].astype(int)

In [ ]:
# topics in df_covid begin with Top_1 but in topics_metadata they begin with Topic 0
df_covid.LDA50_Topic = df_covid.LDA50_Topic.apply(lambda x: x.replace('Top_', 'Topic '))

In [ ]:
topics_metadata = pd.read_csv('data/covidlda_50_metadata_checked.csv', 
                              sep = ';', 
                              encoding = 'utf-8', 
                              quoting=csv.QUOTE_NONNUMERIC)
                              
print(topics_metadata.shape)

In [ ]:
# change the numeric topic to match with df_covid so make Topic n+1
topics_metadata['topic'] = topics_metadata['topic'].apply(lambda x: f'Topic {int(x.split(" ")[1]) + 1}')

In [ ]:
# merge df_covid with topics_metadata to get the topic words and top paragraphs
df_covid_merged = df_covid.merge(topics_metadata, left_on='LDA50_Topic', right_on='topic', how='left')
print(df_covid_merged.shape)

In [ ]:
# limit the df df_covid_merged to January 2020 and May 2022
df_covid_merged['Date'] = pd.to_datetime(df_covid_merged['Date'])
df_covid_merged = df_covid_merged[(df_covid_merged['Date'] >= '2020-01-01') & (df_covid_merged['Date'] <= '2022-05-31')]
print(df_covid_merged.shape)

In [ ]:
# drop new_name and words
df_covid_merged = df_covid_merged.drop(columns=['topic'])
print(df_covid_merged.shape)

In [ ]:
print(df_covid_merged.shape)
# get topic occurrence percentages without Noise/Incoherent topic
df_covid_cleaned = df_covid_merged[df_covid_merged['new_name_l2'].str.contains('Noise') == False]
print(df_covid_cleaned.shape)

In [ ]:
# save df_covid_cleaned
df_covid_cleaned.to_csv('data/df_covid_paragraphs_with_topics_50_enhanced.csv', sep = ';', encoding = 'utf-8', quoting=csv.QUOTE_NONNUMERIC, index=False)

In [ ]:
percentages = df_covid_merged.new_name_l2.value_counts(dropna=False, sort=True, normalize=True).reset_index()
percentages.columns = ['new_name_l2', 'topic_percentage']

In [ ]:
percentages = df_covid_cleaned.new_name_l2.value_counts(dropna=False, sort=True, normalize=True).reset_index()
percentages.columns = ['new_name_l2', 'topic_percentage']

In [ ]:
nr_topics=df_covid_cleaned['new_name_l2'].nunique()
print(nr_topics)

In [ ]:
percentages['sqrd_percentage'] = percentages['topic_percentage'] ** 2
simpson_div = 1 - np.sum(percentages.sqrd_percentage)
std_simpson_div = simpson_div/((nr_topics-1)/nr_topics)
print(simpson_div, std_simpson_div)

In [ ]:
import pytz
df_covid_cleaned['date'] = pd.to_datetime(df_covid_cleaned['Date'], format='%Y-%m-%d')
df_covid_cleaned['date_local'] = df_covid_cleaned['Date'].dt.tz_localize(pytz.timezone('Europe/Amsterdam'))
df_covid_cleaned['year_week'] = df_covid_cleaned['date_local'].dt.strftime('%G-%V')
df_covid_cleaned['year_month'] = df_covid_cleaned['date_local'].dt.strftime('%Y-%m')

In [ ]:
# calculate the monthly occurrence of each topic
monthly_topic_counts = df_covid_cleaned.groupby(['year_month', 'new_name_l2']).size().reset_index(name='count')
total_paragraphs_per_month = df_covid_cleaned.groupby('year_month').size().reset_index(name='total_count')
monthly_topic_counts = monthly_topic_counts.merge(total_paragraphs_per_month, on='year_month', how='left')
monthly_topic_counts['monthly_percentage'] = (monthly_topic_counts['count'] / monthly_topic_counts['total_count'])

In [ ]:
# calculate the weekly occurrence of each topic
weekly_topic_counts = df_covid_cleaned.groupby(['year_week', 'new_name_l2']).size().reset_index(name='count')
total_paragraphs_per_week = df_covid_cleaned.groupby('year_week').size().reset_index(name='total_count')
weekly_topic_counts = weekly_topic_counts.merge(total_paragraphs_per_week, on='year_week', how='left')
weekly_topic_counts['weekly_percentage'] = (weekly_topic_counts['count'] / weekly_topic_counts['total_count'])

In [ ]:
# create a line plot for each topic showing the monthly percentage over time
fig, ax = plt.subplots(figsize=(14, 8))   # increase width so plot area stays wide

unique_topics = monthly_topic_counts['new_name_l2'].unique()
n_topics = len(unique_topics)

# choose a colormap that can produce n distinct colors
if n_topics <= 20:
    cmap = plt.cm.get_cmap('tab20', n_topics)
else:
    cmap = plt.cm.get_cmap('hsv', n_topics)

colors = [cmap(i) for i in range(n_topics)]
color_map = dict(zip(unique_topics, colors))

lines = []
labels = []
for topic in unique_topics:
    topic_data = monthly_topic_counts[monthly_topic_counts['new_name_l2'] == topic]
    line, = ax.plot(topic_data['year_month'].astype(str),
                    topic_data['monthly_percentage'],
                    marker='o',
                    label=topic,
                    color=color_map[topic],
                    linewidth=1)
    lines.append(line)
    labels.append(topic)

ax.set_xlabel('Year-Month')
ax.set_ylabel('Monthly Percentage of Paragraphs')
ax.set_title('Monthly Percentage of Topics Over Time')
ax.tick_params(axis='x', rotation=45)

# place a single figure-level legend below the axes so the axes width is not shrunk
ncol = min(max(1, int(np.ceil(n_topics / 6))), n_topics)
fig.legend(lines, labels, loc='lower center', bbox_to_anchor=(0.5, -0.12),
           ncol=ncol, fontsize='small', frameon=False)

# leave explicit room at bottom; tight_layout with rect keeps axes width
plt.tight_layout(rect=[0, 0.06, 1, 1])
plt.show()

In [ ]:
# for each month, calculate the topic percentages
monthly_percentages = monthly_topic_counts.groupby('year_month').apply(
    lambda x: x.assign(monthly_percentage=x['count'] / x['count'].sum())
).reset_index(drop=True)

In [ ]:
# for each week, calculate the topic percentages
weekly_percentages = weekly_topic_counts.groupby('year_week').apply(
    lambda x: x.assign(weekly_percentage=x['count'] / x['count'].sum())
).reset_index(drop=True)

In [ ]:
# check whether monthly percentages sum to 1 for each month
check = monthly_percentages.groupby('year_month')['monthly_percentage'].sum().reset_index()
print(check.monthly_percentage.unique())

In [ ]:
nr_topics=df_covid_cleaned['new_name_l2'].nunique()

In [ ]:
# calculate simpson diversity index for each month
simpson_indices = []
std_simpson_indices = []
for month, group in monthly_percentages.groupby('year_month'):
    percentages = group['monthly_percentage']
    simpson_div = 1 - np.sum(percentages ** 2)
    nr_topics = (percentages > 0).sum()
    std_simpson_div = simpson_div / ((nr_topics - 1) / nr_topics)
    std_simpson_indices.append({'year_month': month, 'std_simpson_index': std_simpson_div})
std_simpson_df = pd.DataFrame(std_simpson_indices)

In [ ]:
# calculate simpson diversity index for each week
simpson_indices = []
std_simpson_indices = []
for week, group in weekly_percentages.groupby('year_week'):
    percentages = group['weekly_percentage']
    simpson_div = 1 - np.sum(percentages ** 2)
    nr_topics = (percentages > 0).sum()
    std_simpson_div = simpson_div / ((nr_topics - 1) / nr_topics)
    std_simpson_indices.append({'year_week': week, 'std_simpson_index': std_simpson_div})
std_simpson_df_week = pd.DataFrame(std_simpson_indices)

In [ ]:
# calculate the unique number of topics per month
unique_topics_per_month = monthly_percentages[monthly_percentages['monthly_percentage'] > 0].groupby('year_month')['new_name_l2'].nunique().reset_index()
unique_topics_per_month.columns = ['year_month', 'nr_unique_topics']

In [ ]:
# calculate the unique number of topics per week
unique_topics_per_week = weekly_percentages[weekly_percentages['weekly_percentage'] > 0].groupby('year_week')['new_name_l2'].nunique().reset_index()
unique_topics_per_week.columns = ['year_week', 'nr_unique_topics']

In [ ]:
# save std_simpson_df
print(std_simpson_df.shape)
std_simpson_df['model'] = 'LDA'
std_simpson_df_week['model'] = 'LDA'
std_simpson_df.to_csv('data/LDA_simpson_diversity_monthly.csv', sep = ';', encoding = 'utf-8', quoting=csv.QUOTE_NONNUMERIC, index=False)
std_simpson_df_week.to_csv('data/LDA_simpson_diversity_weekly.csv', sep = ';', encoding = 'utf-8', quoting=csv.QUOTE_NONNUMERIC, index=False)

# Read manually annotated dataset

In [ ]:
manual_df = pd.read_csv('data/coded_df_topics_full.csv', 
                        sep=';', encoding='utf-8', quoting=csv.QUOTE_NONNUMERIC)
manual_df = manual_df[manual_df['about_covid'] == 1]
manual_df['article_id'] = manual_df['article_id'].astype(int)
topic_vars = ['about_covid',  'topic_a', 'topic_b', 'topic_c', 'topic_d', 'topic_e', 'topic_f', 'topic_g', 'topic_h', 
              'topic_i', 'topic_j', 'topic_k', 'topic_l', 'topic_m', 'topic_n']

# change all topic vars to int
for i in topic_vars:
    manual_df[i] = manual_df[i].astype(int)

# combine topic_i and topic_j to topic_ij if one is 1 then topic_ij is 1
manual_df['topic_ij'] = manual_df[['topic_i', 'topic_j']].max(axis=1)
manual_df = manual_df.drop(columns=['topic_i', 'topic_j'])

print(manual_df.shape)

In [ ]:
manual_df = manual_df[manual_df['coder'].isin(['main_coder'])]
manual_df = manual_df[manual_df['reliability_article'] == 0]
print(manual_df.shape)

In [ ]:
manual_df = manual_df.rename(columns={'topic_a': 'Pandemic Statistics and Status Updates',
                                                      'topic_b': 'Covid-19 Restrictions and Measures',
                                                      'topic_c': 'Covid-19 Tests and Testing Procedures',
                                                      'topic_d': 'Covid-19 Vaccines, Vaccination Procedures and Campaigns (incl. 2G & 3G)',
                                                      'topic_e': 'Long-Covid and Long-Term Effects of Covid-19 on Health',
                                                      'topic_f': 'Healthcare, Medical Response and Challenges',
                                                      'topic_g': 'Scientific/Medical Research/Knowledge on Covid-19 and the Coronavirus',
                                                      'topic_h': 'Impact of the Pandemic on Economy & Recovery Measures',
                                                      'topic_ij': 'Societal Consequences of the Pandemic & Mental Health',
                                                      'topic_k': 'Impact of Pandemic on Rights and Liberties',
                                                      'topic_l': 'Misinformation about the coronavirus and pandemic & conspiracy theories',
                                                      'topic_m': 'Impact of Pandemic on Politics and Political Discussions',
                                                      'topic_n': 'Global Response & International Collaboration ',
                                                      })


 
subtopics = ['Pandemic Statistics and Status Updates',
        'Covid-19 Restrictions and Measures',
        'Covid-19 Tests and Testing Procedures',
        'Covid-19 Vaccines, Vaccination Procedures and Campaigns (incl. 2G & 3G)',
        'Long-Covid and Long-Term Effects of Covid-19 on Health',
        'Healthcare, Medical Response and Challenges',
        'Scientific/Medical Research/Knowledge on Covid-19 and the Coronavirus',
        'Impact of the Pandemic on Economy & Recovery Measures',
        'Societal Consequences of the Pandemic & Mental Health',
        'Misinformation about the coronavirus and pandemic & conspiracy theories',
        'Impact of Pandemic on Rights and Liberties',
        'Impact of Pandemic on Politics and Political Discussions',
        'Global Response & International Collaboration ']
manual_df_filtered = manual_df[manual_df['article_id'].isin(df_covid_merged['article_id'])]
nr_articles_total = manual_df_filtered['article_id'].nunique()

In [ ]:
print(nr_articles_total)

In [ ]:
subtopics_counts = manual_df_filtered[subtopics].sum()
subtopics_counts = subtopics_counts.sort_values(ascending=False)
subtopics_counts = subtopics_counts.reset_index()
subtopics_counts.columns = ['Subtopic', 'n_articles']
subtopics_counts['percentage'] = (subtopics_counts['n_articles'] / nr_articles_total)

In [ ]:
subtopics_counts['percentage'] = subtopics_counts['percentage'] * 100
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
# Set global parameters for APA 7 compliance
mpl.rcParams['figure.dpi'] = 300
plt.rcParams['font.family'] = 'serif'  # APA recommends serif fonts
plt.rcParams['font.size'] = 12  # Base font size for APA

# Use appropriate figure size for APA (6.5 x 8 inches for Word document)
plt.figure(figsize=(6.5, 8))
sns.barplot(data=subtopics_counts, x='percentage', y='Subtopic', palette='rocket')
plt.xlim(0, 50)

# Add percentage labels
for i, v in enumerate(subtopics_counts['percentage']):
    plt.text(v + 0.5, i, str(round(v, 1)) + '%', color='black', fontsize=12, va='center')

# Format axes
plt.xticks(fontsize=12)
plt.yticks(fontsize=12, rotation=0, ha='right')
plt.xlabel('')
plt.ylabel('')
plt.title('Sub-topic Distribution in Annotated Articles')
plt.xlim(0, 100)
plt.tight_layout()

In [ ]:
df_covid_filtered = df_covid_cleaned[df_covid_merged['article_id'].isin(manual_df['article_id'])]
print(df_covid_filtered.shape)

In [ ]:
# pivot the df to have article_id as index and new_name_l2 as columns with values 1 if the topic is present in the article, else 0
df_pivot = df_covid_filtered.pivot_table(index='article_id',
                                         columns='new_name_l2',
                                         values='paragraphs',
                                         aggfunc='nunique',
                                         fill_value=0)


# make it binary for each topic, if count > 0 then 1 else 0
df_pivot = df_pivot.applymap(lambda x: 1 if x > 0 else 0)
print(df_pivot.shape)

In [ ]:
unique_nr_articles = df_covid_filtered['article_id'].nunique()

In [ ]:
subtopics_counts_LDA = df_pivot.sum().sort_values(ascending=False).reset_index()
subtopics_counts_LDA.columns = ['Subtopic', 'n_articles']
subtopics_counts_LDA['percentage'] = (subtopics_counts_LDA['n_articles'] / unique_nr_articles)

In [ ]:
subtopics_counts_LDA['percentage'] = subtopics_counts_LDA['percentage'] * 100
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
# Set global parameters for APA 7 compliance
mpl.rcParams['figure.dpi'] = 300
plt.rcParams['font.family'] = 'serif'  # APA recommends serif fonts
plt.rcParams['font.size'] = 12  # Base font size for APA

# Use appropriate figure size for APA (6.5 x 8 inches for Word document)
plt.figure(figsize=(6.5, 8))
sns.barplot(data=subtopics_counts_LDA, x='percentage', y='Subtopic', palette='rocket')
plt.xlim(0, 50)

# Add percentage labels
for i, v in enumerate(subtopics_counts_LDA['percentage']):
    plt.text(v + 0.5, i, str(round(v, 1)) + '%', color='black', fontsize=12, va='center')

# Format axes
plt.xticks(fontsize=12)
plt.yticks(fontsize=12, rotation=0, ha='right')
plt.xlabel('')
plt.ylabel('')
plt.title('Sub-topic Distribution in LDA Classified Articles')

# make it go from 0 to 100 on x axis
plt.xlim(0, 100)

plt.tight_layout()

In [ ]:
# get the monthly unique topics from df_covid_filtered
monthly_counts_lda = df_covid_filtered.groupby('year_month')['new_name_l2'].nunique().reset_index()
monthly_counts_lda.columns = ['year_month', 'nr_unique_topics']

In [ ]:
# get the weekly unique topics from df_covid_filtered
weekly_counts_lda = df_covid_filtered.groupby('year_week')['new_name_l2'].nunique().reset_index()
weekly_counts_lda.columns = ['year_week', 'nr_unique_topics']